In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [5]:
X = df.drop("Class" , axis = 1)
y = df["Class"]

In [6]:
df["Class"].unique() # --> means in output layer we have 7 categorical values

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
# 1. 1-2 hidden layers
# 2. 32 neurons , 64 neurons and so on
# If underfitting case than increase no. of neurons



In [8]:
from sklearn.preprocessing import StandardScaler , LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()

y = le.fit_transform(y)

X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42
)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset

In [11]:
X_train_tensor = torch.tensor(X_train_scaled , dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled , dtype = torch.float32)

y_train_tensor = torch.tensor(y_train , dtype = torch.long)
y_test_tensor = torch.tensor(y_test , dtype = torch.long)

In [12]:
train_dataset = TensorDataset(X_train_tensor , y_train_tensor)
test_dataset = TensorDataset(X_test_tensor , y_test_tensor)

In [14]:
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle = True)
test_loader = DataLoader(test_dataset , batch_size = 32 )

In [20]:
# Build our Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN , self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1] , 64),
            nn.ReLU(),
            nn.Linear(64 , 64),
            nn.ReLU(),
            nn.Linear(64 , 7),
        )

    def forward(self , x):
        return self.model(x)
    

In [21]:
model = ANN()

# loss & optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [22]:
# Training the ANN

epochs = 100
for epoch in range (epochs):
    model.train()

    running_loss = 0.0

    for xb , yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs , yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()
        
    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch + 1}/{epochs} , loss = {train_loss}")

epoch = 1/100 , loss = 1.7172210527502971
epoch = 2/100 , loss = 1.1645032426585322
epoch = 3/100 , loss = 0.8087650822556537
epoch = 4/100 , loss = 0.5997850221136342
epoch = 5/100 , loss = 0.49224888241809345
epoch = 6/100 , loss = 0.4318370106427566
epoch = 7/100 , loss = 0.3902174720297689
epoch = 8/100 , loss = 0.34278952038806415
epoch = 9/100 , loss = 0.31143819573132886
epoch = 10/100 , loss = 0.29405028923698096
epoch = 11/100 , loss = 0.27122232965801074
epoch = 12/100 , loss = 0.2489508409862933
epoch = 13/100 , loss = 0.2307843540025794
epoch = 14/100 , loss = 0.21310267824193704
epoch = 15/100 , loss = 0.2077960773654606
epoch = 16/100 , loss = 0.19831041181864945
epoch = 17/100 , loss = 0.18680693954229355
epoch = 18/100 , loss = 0.1801112504756969
epoch = 19/100 , loss = 0.1723390937499378
epoch = 20/100 , loss = 0.16774213605600855
epoch = 21/100 , loss = 0.16588548246933066
epoch = 22/100 , loss = 0.16004427068907282
epoch = 23/100 , loss = 0.14784019148868063
epoch = 

In [25]:
# Evaluate
model.eval() # this step is imp so that model is in evaluation mode no further learning is done by model

total = 0
correct = 0

with torch.no_grad():
    for xb , yb in test_loader:
        outputs = model(xb)  # output [0.2 , 0.3 , 0.5 ...]
        _ , predicted = torch.max(outputs , 1) # --> max value and its index

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual sample in each batch

print(f"total values : {total} & correct values : {correct}")
print(f"accuracy : {correct / total}")

total values : 180 & correct values : 170
accuracy : 0.9444444444444444
